# Programowanie w języku Python

## Wykład 12 — Iteratory, generatory, comprehensions i dekoratory

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć różnicę między obiektem iterowalnym i iteratorem,
- wiedzieć, jak działa pętla `for`,
- korzystać z `iter()` i `next()`,
- tworzyć generatory przez `yield`,
- rozumieć leniwe przetwarzanie danych,
- korzystać z generator expressions,
- stosować list/dict/set comprehensions,
- wiedzieć, kiedy comprehension pogarsza czytelność,
- rozumieć mechanizm dekoratora,
- poprawnie zachowywać metadane funkcji przez `functools.wraps`.

## Pętla `for` nie potrzebuje indeksu

Na W3 poznaliśmy:

```python
for value in values:
    print(value)
```

Python nie musi używać indeksów.

Pętla potrzebuje obiektu, po którym można **iterować**.

## Iterable — obiekt iterowalny

Przykłady obiektów iterowalnych:

```text
list
tuple
str
dict
set
range
plik
generator
```

Obiekt iterowalny potrafi dostarczyć iterator.

## Iterator

Iterator dostarcza kolejne elementy sekwencji.

Możemy utworzyć go ręcznie:

```python
iterator = iter(values)
```

a następnie pobierać kolejne wartości:

```python
next(iterator)
```

In [ ]:
values = [10, 20, 30]

iterator = iter(values)

print(next(iterator))
print(next(iterator))
print(next(iterator))

## Koniec iteratora

Po wyczerpaniu elementów:

```python
next(iterator)
```

zgłasza:

```text
StopIteration
```

Pętla `for` obsługuje ten mechanizm automatycznie.

In [ ]:
values = [10, 20]

iterator = iter(values)

print(next(iterator))
print(next(iterator))

try:
    print(next(iterator))
except StopIteration:
    print("Koniec danych")

## Uproszczony model pętli `for`

Kod:

```python
for value in values:
    print(value)
```

możemy rozumieć w przybliżeniu jako:

```python
iterator = iter(values)

while True:
    try:
        value = next(iterator)
    except StopIteration:
        break

    print(value)
```

## Iterator jest obiektem stanowym

Iterator „pamięta”, gdzie zakończył poprzednie odczytanie.

```text
10 → 20 → 30 → StopIteration
      ↑
   bieżąca pozycja
```

To ważne: iterator może zostać **wyczerpany**.

In [ ]:
values = [1, 2, 3]
iterator = iter(values)

print(list(iterator))
print(list(iterator))

## Iterable a iterator

```text
ITERABLE
↓ iter(...)
ITERATOR
↓ next(...)
kolejne wartości
```

Lista jest iterowalna, ale sama nie jest typowym jednorazowym iteratorem.

Generator jest jednocześnie iteratorem.

## Po co generatory?

Załóżmy, że chcemy utworzyć milion wartości:

```python
values = [i * i for i in range(1_000_000)]
```

Lista przechowuje wszystkie wyniki naraz.

A jeśli potrzebujemy tylko kolejnych wartości, po jednej?

## Funkcja generatorowa

Jeżeli funkcja zawiera:

```python
yield
```

to jej wywołanie tworzy generator.

In [ ]:
def count_to_three():
    yield 1
    yield 2
    yield 3

generator = count_to_three()

print(generator)
print(type(generator))

## `yield` a `return`

`return` kończy zwykłą funkcję.

`yield`:

- zwraca kolejną wartość,
- zatrzymuje wykonanie funkcji,
- zapamiętuje jej stan,
- pozwala później kontynuować od tego miejsca.

In [ ]:
def count_to_three():
    print("start")
    yield 1

    print("po pierwszym yield")
    yield 2

    print("po drugim yield")
    yield 3

generator = count_to_three()

print(next(generator))
print(next(generator))
print(next(generator))

## Generator i pętla `for`

Najczęściej nie wywołujemy `next()` ręcznie.

```python
for value in count_to_three():
    print(value)
```

Pętla obsługuje iterator generatora automatycznie.

In [ ]:
for value in count_to_three():
    print("otrzymano:", value)

## Praktyczny generator

Wygenerujmy kolejne poprawne pomiary:

In [ ]:
def valid_measurements(values):
    for value in values:
        if value >= -273.15:
            yield value

measurements = [21.7, -300, 22.1, 35.0]

for value in valid_measurements(measurements):
    print(value)

## Leniwe przetwarzanie

Generator nie musi tworzyć wszystkich wyników od razu.

```text
wejście
↓
pobierz element
↓
przetwórz
↓
yield
↓
następny element
```

To jest **lazy evaluation** — obliczamy wartość dopiero, gdy jest potrzebna.

## Generator dla dużych danych

Przykład:

```python
def squares(n):
    for i in range(n):
        yield i * i
```

Generator może reprezentować bardzo długi strumień wyników bez przechowywania wszystkich wartości jednocześnie.

In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

generator = squares(1_000_000)

print(next(generator))
print(next(generator))
print(next(generator))

## Generator expression

Podobnie do list comprehension:

```python
[i * i for i in range(10)]
```

możemy utworzyć generator:

```python
(i * i for i in range(10))
```

In [ ]:
list_result = [i * i for i in range(5)]
generator_result = (i * i for i in range(5))

print(list_result)
print(generator_result)
print(list(generator_result))

## Lista czy generator?

```text
LISTA
↓
wszystkie wartości istnieją od razu
↓
możemy wielokrotnie iterować
↓
większe zużycie pamięci

GENERATOR
↓
wartości powstają na żądanie
↓
zwykle jednorazowa iteracja
↓
małe zużycie pamięci
```

## Pomiar pamięci — ostrożnie z interpretacją

Możemy porównać wielkość samych obiektów reprezentujących kolekcję:

In [ ]:
import sys

list_values = [i * i for i in range(100_000)]
generator_values = (i * i for i in range(100_000))

print("list:", sys.getsizeof(list_values))
print("generator:", sys.getsizeof(generator_values))

## Co ten pomiar pokazuje?

`sys.getsizeof()` nie mierzy całkowitego kosztu całej struktury i wszystkich obiektów potomnych.

Ale dobrze pokazuje ideę:

> generator nie przechowuje 100 000 gotowych wyników.

Nie zapamiętujemy konkretnej liczby bajtów — rozumiemy mechanizm.

## List comprehension

Klasyczna pętla:

```python
squares = []

for value in range(10):
    squares.append(value ** 2)
```

może zostać zapisana:

```python
squares = [value ** 2 for value in range(10)]
```

In [ ]:
squares = [value ** 2 for value in range(10)]

print(squares)

## Comprehension jako transformacja

Czytajmy:

```python
[value ** 2 for value in values]
```

jako:

```text
dla każdego value w values
↓
oblicz value ** 2
↓
dodaj wynik do nowej listy
```

## Filtrowanie

Możemy dodać warunek:

```python
[value for value in values if value >= 0]
```

In [ ]:
values = [-3, -1, 0, 2, 5]

non_negative = [
    value
    for value in values
    if value >= 0
]

print(non_negative)

## Transformacja z warunkiem

To inna konstrukcja:

```python
[value if value >= 0 else 0 for value in values]
```

Tutaj **każdy element** daje wynik, ale sposób transformacji zależy od warunku.

In [ ]:
values = [-3, -1, 0, 2, 5]

clipped = [
    value if value >= 0 else 0
    for value in values
]

print(clipped)

## Dict comprehension

```python
{name: value for ...}
```

Przykład:

In [ ]:
temperatures = [21.7, 22.1, 20.8]

indexed = {
    index: value
    for index, value in enumerate(temperatures, start=1)
}

print(indexed)

## Set comprehension

```python
{expression for item in iterable}
```

tworzy zbiór:

In [ ]:
words = ["Python", "python", "PYTHON", "Java"]

languages = {
    word.casefold()
    for word in words
}

print(languages)

## Kiedy comprehension jest dobry?

Dobrze sprawdza się przy:

- prostej transformacji,
- prostym filtrowaniu,
- tworzeniu nowej kolekcji.

Jeżeli logika wymaga wielu warunków, efektów ubocznych i kilku kroków — zwykła pętla często jest czytelniejsza.

## Czy comprehension jest zawsze szybszy?

Często bywa szybszy od ręcznej pętli z `append()`, ale:

> najpierw wybieramy rozwiązanie czytelne i poprawne.

Optymalizację wykonujemy wtedy, gdy pomiar pokazuje realny problem.

In [ ]:
values = list(range(1000))

%timeit [value * value for value in values]

In [ ]:
%%timeit
result = []
for value in values:
    result.append(value * value)

## Funkcje jako obiekty — przypomnienie W5

Potrafimy już:

```python
def square(x):
    return x * x

function = square
```

oraz przekazywać funkcję jako argument.

To jest fundament dekoratorów.

## Funkcja może zwracać funkcję

```python
def outer():

    def inner():
        print("inner")

    return inner
```

W W5 nazwaliśmy taki mechanizm podstawą closures.

## Dekorator — idea

Dekorator to obiekt wywoływalny, który przyjmuje funkcję lub klasę i zwraca jej zmodyfikowaną albo opakowaną wersję.

Najprostszy przypadek:

```text
funkcja
↓
dekorator
↓
nowa funkcja
```

In [ ]:
def add_logging(function):

    def wrapper():
        print("Przed wywołaniem")
        function()
        print("Po wywołaniu")

    return wrapper


def hello():
    print("Hello")

hello = add_logging(hello)

hello()

## Składnia z `@`

Zapis:

```python
@add_logging
def hello():
    print("Hello")
```

jest równoważny idei:

```python
hello = add_logging(hello)
```

In [ ]:
def add_logging(function):

    def wrapper():
        print("Przed")
        function()
        print("Po")

    return wrapper


@add_logging
def hello():
    print("Hello")


hello()

## Dekorator dla funkcji z argumentami

Wrapper powinien przekazać argumenty dalej:

In [ ]:
def trace(function):

    def wrapper(*args, **kwargs):
        print("args:", args)
        print("kwargs:", kwargs)
        return function(*args, **kwargs)

    return wrapper


@trace
def add(a, b):
    return a + b


print(add(2, 3))

## Wrapper powinien zwracać wynik

Jeśli dekorowana funkcja zwraca wartość, wrapper również powinien ją zwrócić:

```python
return function(*args, **kwargs)
```

Inaczej zmienimy kontrakt funkcji.

## Problem z metadanymi

Prosty dekorator ukrywa m.in.:

- nazwę funkcji,
- docstring,
- część informacji używanych przez narzędzia.

Sprawdźmy:

In [ ]:
print(add.__name__)

## `functools.wraps`

Poprawny wzorzec:

In [ ]:
from functools import wraps


def trace(function):

    @wraps(function)
    def wrapper(*args, **kwargs):
        print(
            f"Wywołanie {function.__name__}"
        )
        return function(*args, **kwargs)

    return wrapper


@trace
def multiply(a, b):
    """Multiply two numbers."""
    return a * b


print(multiply.__name__)
print(multiply.__doc__)

## Praktyczny dekorator: pomiar czasu

Dekorator może dodać zachowanie wspólne dla wielu funkcji:

In [ ]:
from functools import wraps
from time import perf_counter


def measure_time(function):

    @wraps(function)
    def wrapper(*args, **kwargs):
        start = perf_counter()

        result = function(*args, **kwargs)

        elapsed = perf_counter() - start

        print(
            f"{function.__name__}: "
            f"{elapsed:.6f} s"
        )

        return result

    return wrapper

In [ ]:
@measure_time
def calculate():
    return sum(i * i for i in range(100_000))

result = calculate()

print(result)

## Dekorator nie powinien być „magią”

Warto zawsze umieć odpowiedzieć:

```text
jaka funkcja wchodzi?
↓
co robi wrapper?
↓
jakie argumenty przekazuje?
↓
co zwraca?
```

Jeżeli dekorator utrudnia zrozumienie programu, prawdopodobnie nie pomaga.

## Dekoratory znamy już z OOP

Na W10–W11 używaliśmy:

```python
@property
@classmethod
@staticmethod
@dataclass
```

Teraz wiemy, że składnia `@` oznacza zastosowanie dekoratora do definiowanego obiektu.

## Przykład integrujący: pipeline

Generator może dostarczać dane leniwie, comprehension może je transformować, a dekorator mierzyć czas:

In [ ]:
@measure_time
def process_measurements(values):
    valid = (
        value
        for value in values
        if value >= -273.15
    )

    return [
        round(value, 1)
        for value in valid
        if value > 30
    ]


values = [21.7, 35.21, -300, 31.04, 20.2]

print(process_measurements(values))

## Ćwiczenie

1. Napisz generator `read_valid(values)`, który pomija wartości ujemne.
2. Użyj generator expression do przeliczenia wartości przez współczynnik.
3. Użyj list comprehension do zebrania tylko wyników większych od 100.
4. Napisz dekorator `count_calls`, który liczy liczbę wywołań funkcji.
5. Zastosuj `@wraps`, aby zachować nazwę i docstring funkcji.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- iterable może dostarczyć iterator przez `iter()`,
- iterator dostarcza kolejne wartości przez `next()`,
- koniec iteracji sygnalizuje `StopIteration`,
- generator używa `yield` i działa leniwie,
- generator expression nie tworzy od razu całej kolekcji,
- comprehensions są dobre dla prostych transformacji i filtrów,
- czytelność jest ważniejsza niż skracanie kodu,
- dekorator opakowuje funkcję lub klasę,
- zapis `@decorator` odpowiada transformacji definiowanego obiektu,
- wrapper powinien przekazywać argumenty i zwracać wynik,
- `functools.wraps` zachowuje metadane funkcji.

## Następny wykład

# Jakość kodu, testowanie i narzędzia projektu

Skoro potrafimy już korzystać z zaawansowanych mechanizmów Pythona, kolejne pytanie brzmi:

> skąd wiemy, że kod jest czytelny, poprawny i łatwy do rozwijania?